[README](README.md) | [Assignment](ASSIGNMENT.md) | [Fixture data](data/README.md) | Notebook

# Building an MCP Interface for Internet Topology Data

This is the single student notebook deliverable. Run it from top to bottom, and keep every
evidence record with your submission. All concepts, contracts, and the full text of the graded
questions are in [ASSIGNMENT.md](ASSIGNMENT.md).

| Part | What you do | Access path | Questions |
| --- | --- | --- | --- |
| 1 | Investigate real ASes' interconnection and geolocation with hand-written SQL | direct read-only SQL against the real teaching snapshot | Q1-Q3 |
| 2 | Use the 4 provided tools through an agent, in this notebook | OpenAI-compatible client-side tool loop + MCP | Q4-Q5 |
| 3 | Build 3 new general-purpose tools, then investigate with all 7 | your code in `src/itdk_mcp/student_tools.py` + the same agent cells | Q6-Q9 |

The two access paths are not interchangeable. **Part 1 only** uses the direct SQL connection to the
real teaching snapshot; it has the full flexibility of hand-written SQL, which is why it carries the
harder questions. **Parts 2 and 3** go through MCP's small, fixed tool vocabulary -- do not add
database drivers, connection strings, or SQL to those cells.

Before starting Jupyter: export the variables from `itdk_mcp_credentials.env` (including
`OPENAI_API_KEY` and `OPENAI_BASE_URL`), copy `db_credentials.env.example` to
`db_credentials.env` and fill in the real teaching-snapshot DSN your instructor provides, and run
`docker compose --env-file itdk_mcp_credentials.env up -d`.

## Complete setup 1 of 3 (do not edit)

These imports and helpers open short authenticated MCP sessions, capture tool arguments and
structured result metadata, and read only CSV files produced by the MCP server. The bearer key is
never displayed.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import math

import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two lat/lon points (Part 1 Q1)."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))

import pandas as pd
from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_URL = os.environ.get("ITDK_MCP_URL", "http://127.0.0.1:8000/mcp/sse")
SNAPSHOT_ID = os.environ.get("ITDK_SNAPSHOT_ID", "nids-itdk-mcp-synthetic-v1")
SERVER_OUTPUT_DIR = Path(os.environ.get("ITDK_SERVER_OUTPUT_DIR", "/app/outputs"))
LOCAL_OUTPUT_DIR = Path(os.environ.get("ITDK_OUTPUT_DIR", "./outputs"))
MASTER_KEY = os.environ.get("MCP_MASTER_KEY", "")
if not MASTER_KEY:
    raise RuntimeError("Export MCP_MASTER_KEY before starting Jupyter; do not paste it into this notebook.")

async def _session_call(action: str, name: str | None = None, arguments: dict[str, Any] | None = None) -> Any:
    headers = {"Authorization": f"Bearer {MASTER_KEY}"}
    async with sse_client(MCP_URL, headers=headers) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            if action == "list":
                return await session.list_tools()
            assert name is not None and arguments is not None
            return await session.call_tool(name, arguments)

async def list_itdk_tools() -> list[dict[str, Any]]:
    result = await _session_call("list")
    return [tool.model_dump(mode="json", by_alias=True) for tool in result.tools]

async def call_itdk_raw(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    result = await _session_call("call", name, arguments)
    return {"snapshot": SNAPSHOT_ID, "tool_name": name, "arguments": dict(arguments), "result": result.model_dump(mode="json", by_alias=True)}

async def call_itdk_tool(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    raw_evidence = await call_itdk_raw(name, arguments)
    payload = raw_evidence["result"]
    if payload.get("isError"):
        raise RuntimeError(f"{name} returned an MCP error: {payload.get('content')!r}")
    metadata = payload.get("structuredContent")
    if not isinstance(metadata, dict):
        raise RuntimeError(f"{name} returned no structuredContent metadata")
    return {"snapshot": SNAPSHOT_ID, "tool_name": name, "arguments": dict(arguments), "metadata": metadata}

def local_csv_path(evidence: dict[str, Any]) -> Path:
    server_path = Path(evidence["metadata"]["file_path"])
    relative_path = server_path.relative_to(SERVER_OUTPUT_DIR)
    return LOCAL_OUTPUT_DIR / relative_path

def load_tool_csv(evidence: dict[str, Any]) -> pd.DataFrame:
    frame = pd.read_csv(local_csv_path(evidence), keep_default_na=False, na_values=[r"\N"])
    expected = int(evidence["metadata"]["row_count"])
    if len(frame) != expected:
        raise AssertionError(f"CSV row count {len(frame)} does not match metadata {expected}")
    return frame

def show_evidence(evidence: dict[str, Any]) -> None:
    print(json.dumps(evidence, indent=2, sort_keys=True))

## Complete setup 2 of 3 (do not edit) -- Part 1 only

Part 1 reads the four ITDK relations directly with SQL, against the **real teaching snapshot** --
a read-only Postgres role on the shared CAIDA ITDK database that your instructor hands out the same
way a prior CAIDA assignment's `db_credentials.env` was. It restricts every node-keyed table to
known routers and keeps only router-to-router links, so `node_id` here is guaranteed to be an
inferred router (see the "Dataset scope" note your instructor distributes with the connection
details) -- that is the only reason this notebook can speak of "routers" interchangeably with
"nodes".

Copy `db_credentials.env.example` to `db_credentials.env` (it is git-ignored) and fill in the real
`ITDK_READ_DSN` your instructor provides -- never a locally-invented one, and never commit the
filled-in file.

> **Scope:** this is real, access-controlled measurement data. Never commit `db_credentials.env`,
> query results, or generated CSVs derived from it. If `sqlalchemy`, `psycopg2-binary`, or
> `python-dotenv` are missing, install them alongside the project's dev extras (for example
> `uv pip install 'sqlalchemy>=2' psycopg2-binary python-dotenv`).

In [ ]:
# Read-only connection to the REAL TEACHING SNAPSHOT, for Task 1 only.
# Precedence: real env var > db_credentials.env (git-ignored, uploaded next to
# this notebook -- see db_credentials.env.example).
from sqlalchemy import create_engine
from dotenv import dotenv_values

DB_CREDS_FILE = Path(os.environ.get("ITDK_DB_CREDS_FILE", "./db_credentials.env"))
_db_creds = dotenv_values(DB_CREDS_FILE)
READ_DSN = os.environ.get("ITDK_READ_DSN") or _db_creds.get("ITDK_READ_DSN")
if not READ_DSN:
    raise RuntimeError(
        "No ITDK_READ_DSN found. Copy db_credentials.env.example to db_credentials.env and fill "
        "in the real teaching-snapshot DSN your instructor provides; there is no local fallback."
    )
if READ_DSN.startswith("postgresql://"):
    try:  # this project ships psycopg 3; fall back to it when psycopg2 is absent
        import psycopg2  # noqa: F401
    except ModuleNotFoundError:
        READ_DSN = READ_DSN.replace("postgresql://", "postgresql+psycopg://", 1)

engine = create_engine(READ_DSN)

# Confirm the session is the read-only snapshot role before running any Task 1 query.
pd.read_sql(
    """
    SELECT current_user,
           current_database(),
           current_setting('transaction_read_only') AS read_only
    """,
    engine,
)

## Complete setup 3 of 3 (do not edit) -- the in-notebook agent

Parts 2 and 3 drive a real agent from this notebook using **NRP Nautilus's OpenAI-compatible
endpoint**, so you are not limited by needing your own Claude subscription or API key. Because that
endpoint has no equivalent of a server-side remote-MCP connector, this notebook plays the role a
connector would otherwise play: it sends the model a standard OpenAI-style `tools` list built from
`list_itdk_tools()`, executes any `tool_calls` the model requests against this MCP server itself
(via `call_itdk_raw`, above), feeds the results back as `role: "tool"` messages, and repeats until
the model stops asking for tools. There is no separate agent process and no desktop app -- and,
unlike a server-side connector, the model itself never needs network access to the MCP server, only
this notebook process does.

Two things this needs, both set in `itdk_mcp_credentials.env`:

- `OPENAI_API_KEY` -- your NRP Nautilus key (or another OpenAI-compatible provider's key). Never
  paste it into this notebook or commit it.
- `OPENAI_BASE_URL` -- the endpoint base URL (for NRP Nautilus, `https://ellm.nrp-nautilus.io/v1`).

`await call_agent_with_mcp_tools(prompt)` returns `(trace, answer)`: the trace is one record per
tool call / tool result pair -- the tool-call record you audit in Q4, Q5, Q8, and Q9.

In [ ]:
# Complete setup: a client-side tool-calling loop against an OpenAI-compatible
# endpoint (NRP Nautilus by default). Unlike a server-side remote-MCP connector,
# this notebook process executes every tool call itself and relays results to
# the model as text -- the model never needs network access to the MCP server.
from openai import AsyncOpenAI

AGENT_MODEL = os.environ.get("ITDK_AGENT_MODEL", "kimi")  # kimi is the most reliable tool-caller tested
MCP_SERVER_NAME = "itdk-mcp"
_OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
_OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or None

if _OPENAI_API_KEY:
    agent_client = AsyncOpenAI(api_key=_OPENAI_API_KEY, base_url=_OPENAI_BASE_URL)
else:
    agent_client = None
    print("OPENAI_API_KEY is not exported; the Part 2/3 agent cells will not run.")


def _mcp_tool_to_openai_tool(tool: dict[str, Any]) -> dict[str, Any]:
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("inputSchema", {"type": "object", "properties": {}}),
        },
    }


async def call_agent_with_mcp_tools(
    prompt: str, *, max_tokens: int = 16384, model: str | None = None, max_turns: int = 12, temperature: float = 0.0
) -> tuple[list[dict[str, Any]], str]:
    """Run one prompt through a client-side tool-calling loop against this MCP server.

    Returns (trace, answer): `trace` is one record per tool call / tool result pair --
    the record you audit in Q4, Q5, Q8, and Q9; `answer` is the model's final text.
    """
    if agent_client is None:
        raise RuntimeError("Export OPENAI_API_KEY before running the agent cells.")

    tool_specs = [_mcp_tool_to_openai_tool(tool) for tool in await list_itdk_tools()]
    messages: list[dict[str, Any]] = [{"role": "user", "content": prompt}]
    trace: list[dict[str, Any]] = []

    for turn in range(max_turns):
        # On the last turn, stop offering tools so a model that keeps wanting to
        # call more of them is forced to answer from what it already has, rather
        # than the cell raising and breaking a full "restart and run all".
        final_turn = turn == max_turns - 1
        response = await agent_client.chat.completions.create(
            model=model or AGENT_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,  # 0 for the most reproducible trace across runs
            messages=messages,
            tools=None if final_turn else tool_specs,
        )
        message = response.choices[0].message
        tool_calls = message.tool_calls or []
        if not tool_calls:
            return trace, message.content or ""

        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {"name": call.function.name, "arguments": call.function.arguments},
                    }
                    for call in tool_calls
                ],
            }
        )

        for call in tool_calls:
            name = call.function.name
            try:
                arguments = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                arguments = {}
            trace.append({"kind": "call", "tool": name, "server": MCP_SERVER_NAME, "arguments": arguments})
            raw = await call_itdk_raw(name, arguments)
            payload = raw["result"]
            is_error = bool(payload.get("isError"))
            texts = [block.get("text", str(block)) for block in (payload.get("content") or [])]
            trace.append({"kind": "result", "tool_use_id": call.id, "is_error": is_error, "content": texts})
            tool_text = "\n".join(texts) if texts else json.dumps(payload.get("structuredContent", {}))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": tool_text})

    raise AssertionError("unreachable: the forced-final turn above always returns")


async def show_agent_run(prompt: str) -> tuple[list[dict[str, Any]], str]:
    """Run one fixed prompt and print its trace and answer for the record."""
    trace, answer = await call_agent_with_mcp_tools(prompt)
    print("PROMPT\n" + prompt + "\n")
    print("TOOL-CALL TRACE")
    print(json.dumps(trace, indent=2, sort_keys=True, default=str))
    print("\nAGENT ANSWER\n" + answer)
    return trace, answer

---

# Part 1 - Investigate real ASes with direct SQL (Q1-Q3)

The four relations are `caida_itdk.itdk_link_endpoints`, `caida_itdk.itdk_node_as`,
`caida_itdk.itdk_node_geolocation`, and `caida_itdk.itdk_router_hostnames` -- the same schema as
Parts 2/3's MCP tools use, but here you write the SQL by hand against the **real teaching
snapshot**, with the full flexibility of hand-written SQL. That is why Part 1 carries the harder,
more open-ended questions: real router-level topology among real, named autonomous systems.

Each section gives you worked queries plus one block to complete by analogy. Fill in the SQL
between the `student_code_start` / `student_code_end` markers, then answer the question in the
markdown cell that follows.

**This is the only part that touches the database directly.** Record the snapshot's release
identifier in your submission so a reader knows exactly which data your answers describe.

### Q1 - Border routers between Level3 (AS3356) and Netflix (AS2906)

A router-level link with one endpoint's node assigned to AS3356 (Level3, a transit ISP) and the
other to AS2906 (Netflix, a content provider) is a **border router pair**: the physical point at
which one AS hands traffic to the other.

In [ ]:
# Worked: every Level3<->Netflix router-level link, with geolocation for both endpoints.
# Build one router-set per AS (each enriched with geolocation via a join on node_id, the
# router-level join key -- see Q1 below for why endpoint_token would be the wrong key), then
# join the two sets on link_id: a link is a Level3<->Netflix link exactly when the same
# link_id appears in both sets.
geo_df = pd.read_sql(
    """
    WITH level3_nodes AS (
        SELECT
            le.link_id,
            le.node_id AS l3_node_id,
            g.latitude AS l3_lat,
            g.longitude AS l3_lon,
            g.city AS l3_city,
            g.region AS l3_region,
            g.country AS l3_country
        FROM caida_itdk.itdk_link_endpoints le
        JOIN caida_itdk.itdk_node_as na ON le.node_id = na.node_id
        JOIN caida_itdk.itdk_node_geolocation g ON le.node_id = g.node_id
        WHERE na.asn = 3356
    ),
    netflix_nodes AS (
        --- student_code_start ------------------------------------------------
        --- YOUR SQL HERE (identical shape to level3_nodes, filtered to
        --- na.asn = 2906, columns aliased nf_node_id / nf_lat / nf_lon / nf_city /
        --- nf_region / nf_country)
        --- student_code_end --------------------------------------------------
    )
    SELECT
        l3.link_id,
        l3.l3_node_id, l3.l3_lat, l3.l3_lon, l3.l3_city, l3.l3_region, l3.l3_country,
        n.nf_node_id, n.nf_lat, n.nf_lon, n.nf_city, n.nf_region, n.nf_country
    FROM level3_nodes l3
    JOIN netflix_nodes n ON l3.link_id = n.link_id;
    """,
    engine,
)
print(f"Links with geolocation for both endpoints: {len(geo_df)}")
geo_df

In [ ]:
# YOUR CODE HERE
# Output 1: geo_df gains distance_km (haversine great-circle distance between the two
#           endpoints, in km) and adjacent (bool, distance_km <= THRESHOLD_KM).
# Output 2: peering_locations -- for the adjacent links only, group by (l3_city,
#           l3_country), count DISTINCT link_id per group, rename to
#           (city, country, link_count), sort by link_count descending.
# Steps:
#   1. Use the haversine() helper defined in Complete setup 1 with
#      geo_df.apply(..., axis=1) to add distance_km.
#   2. THRESHOLD_KM = 40; adjacent = geo_df["distance_km"] <= THRESHOLD_KM.
#   3. adjacent_df = geo_df[geo_df["adjacent"]]; non_adjacent_df = geo_df[~geo_df["adjacent"]].
#   4. peering_locations from adjacent_df, grouped by (l3_city, l3_country), COUNT(DISTINCT
#      link_id), sorted descending -- pandas groupby + nunique, not SQL, since this operates
#      on the DataFrame you already pulled.
THRESHOLD_KM = 40
# student_code_start ------------------------------------------------
# YOUR CODE HERE
# student_code_end --------------------------------------------------

num_adjacent = len(adjacent_df)
num_non_adjacent = len(non_adjacent_df)
print(f"Adjacent links     (<= {THRESHOLD_KM} km): {num_adjacent}")
print(f"Non-adjacent links  (> {THRESHOLD_KM} km): {num_non_adjacent}")
print("\nNon-adjacent link IDs:", sorted(non_adjacent_df["link_id"].unique()))
print(f"\nDistinct peering locations (adjacent links): {len(peering_locations)}")
peering_locations

**Q1** (a) How many links came out geographically adjacent versus not, and what are the link IDs
of the non-adjacent ones? (b) Excluding the non-adjacent links, at how many distinct locations did
Level3 and Netflix appear to peer, and what is the distribution across those locations (present
`peering_locations`)? (c) Write a paragraph interpreting the peering relationship -- focus on the
economics and performance reasons an ISP and a content provider would prefer to co-locate their
interconnection, and on whether the non-adjacent outliers look like real long-haul links or
geolocation artifacts (a Hoiho-geolocated endpoint paired with a Maxmind-geolocated one is a common
source of that kind of error).

*Your answer for Q1:*

Replace this line with your answer.

### Q2 - Router concentration: China Unicom (AS4837) vs. Level3 (AS3356)

China Unicom operates a national backbone serving mostly domestic Chinese users; Level3 is a
global transit ISP. Their geolocated router footprints should look very different -- this section
quantifies exactly how.

In [ ]:
# Worked: geolocated router count per (ASN, country) for both ASes in one round trip.
country_counts = pd.read_sql(
    """
    SELECT
        na.asn,
        g.country,
        COUNT(DISTINCT na.node_id) AS node_count
    FROM caida_itdk.itdk_node_as na
    JOIN caida_itdk.itdk_node_geolocation g ON g.node_id = na.node_id
    WHERE na.asn IN (4837, 3356)
    GROUP BY na.asn, g.country
    """,
    engine,
)

def _by_country(asn):
    t = country_counts[country_counts["asn"] == asn].set_index("country")[["node_count"]].copy()
    t["rank"] = t["node_count"].rank(method="min", ascending=False).astype(int) - 1  # 0 = most
    t["pct"] = 100 * t["node_count"] / t["node_count"].sum()
    return t.sort_values("node_count", ascending=False)

cu_by_country = _by_country(4837)
l3_by_country = _by_country(3356)
print(f"Total geolocated China Unicom (AS4837) nodes: {int(cu_by_country['node_count'].sum())}")
print(f"Total geolocated Level3 (AS3356) nodes: {int(l3_by_country['node_count'].sum())}")
print(f"China Unicom country count: {len(cu_by_country)}; Level3 country count: {len(l3_by_country)}")

# YOUR CODE HERE
# Output: comparison -- China Unicom's top 10 countries by router count, with columns
#         country_name, cu_num_router, cu_rank, cu_pct, l3_pct, l3_rank, l3_num_routers.
#         A country with zero Level3 routers gets l3_pct = 0.0 and l3_rank below every
#         ranked country (use len(l3_by_country) as that fallback rank).
# Steps:
#   1. Map each two-letter country code to a name with pycountry.countries.get(alpha_2=code).name
#      (wrap in try/except -- a handful of codes may not resolve).
#   2. For each of cu_by_country.head(10)'s countries, look up the matching row in
#      l3_by_country if present, else use the zero/fallback values above.
#   3. Assemble the rows into a DataFrame named comparison.
# student_code_start ------------------------------------------------
# YOUR SQL/CODE HERE
# student_code_end --------------------------------------------------
print("\nAS4837 top 10 countries, compared against Level3 in the same countries:")
comparison

**Q2** (a) How does China Unicom's country concentration compare with Level3's, and what about
their business models explains the difference? (b) Which AS has more total routers, and why might
that be the case given their different roles?

*Your answer for Q2 (a, b):*

Replace this line with your answer.

#### China Unicom's West Coast ASN peering

China Unicom's US West Coast presence is concentrated in Los Angeles and San Jose, the western
end of trans-Pacific submarine cables. For each West Coast China Unicom router, find which other
ASes connect to it at the router level, using both AS assignment data and interface hostnames.

In [ ]:
# Worked: for each West Coast China Unicom (AS4837) router, find every peer AS sharing a
# router-level link with it, with the peer interface's hostname where one exists.
cu_peers_df = pd.read_sql(
    """
    WITH cu AS (
        -- seed set: West Coast China Unicom routers (AS4837, US, west of -115 lon)
        SELECT a.node_id, g.city
        FROM caida_itdk.itdk_node_as a
        JOIN caida_itdk.itdk_node_geolocation g ON g.node_id = a.node_id
        WHERE a.asn = 4837 AND g.country = 'US' AND g.longitude < -115
    )
    SELECT DISTINCT
        cu.node_id  AS cu_node,
        cu.city     AS cu_city,
        e2.node_id  AS peer_node,
        a2.asn      AS peer_asn,
        h.hostname  AS peer_hostname
    FROM cu
    JOIN caida_itdk.itdk_link_endpoints e1 ON e1.node_id = cu.node_id
    JOIN caida_itdk.itdk_link_endpoints e2
        ON e2.link_id = e1.link_id AND e2.node_id <> cu.node_id
    JOIN caida_itdk.itdk_node_as a2
        ON a2.node_id = e2.node_id AND a2.asn <> 4837
    LEFT JOIN caida_itdk.itdk_router_hostnames h
        ON h.ip = NULLIF(split_part(e2.endpoint_token, ':', 2), '')::inet
    """,
    engine,
)
print(f"West Coast CU router-peer rows: {len(cu_peers_df)}")
print(f"Distinct west-coast CU routers with a peer: {cu_peers_df['cu_node'].nunique()}")
print(f"Distinct peer ASNs: {cu_peers_df['peer_asn'].nunique()}")

# YOUR CODE HERE
# Output: city_peer_table -- for the top 10 West Coast cities by distinct CU router count,
#         columns (total, <peer ASN columns...>). `total` = distinct CU routers per city
#         (NOT the sum of the peer columns -- a router with several peers counts once).
# Steps:
#   1. d = cu_peers_df.drop_duplicates(subset=["cu_city", "cu_node", "peer_asn"]) so a router
#      with several links to the SAME peer is not double-counted.
#   2. city_total = d.groupby("cu_city")["cu_node"].nunique(), sorted descending, top 10 cities.
#   3. wide = d[d.cu_city.isin(top_cities)].groupby(["cu_city","peer_asn"])["cu_node"].nunique()
#      .unstack(fill_value=0) -- one column per peer ASN, one row per city.
#   4. city_peer_table = wide with a leading "total" column from city_total.
# student_code_start ------------------------------------------------
# YOUR CODE HERE
# student_code_end --------------------------------------------------
print(f"\nTop West Coast China Unicom cities: {len(city_peer_table)}")
city_peer_table

**Q2** (c) Present the West Coast China Unicom peering table (`city_peer_table`). (d) Write a
paragraph: why would China Unicom connect to the **same** peer AS more than once at the **same**
city, and why would it connect to the same peer AS at **different** cities?

*Your answer for Q2 (c, d):*

Replace this line with your answer.

### Q3 - Interconnection structure across 18 major ASes

The table below lists 18 major ASes grouped by category (transit backbone first, content last):

| ASN   | Name              | Code | Description |
|-------|-------------------|------|--------------|
| 174   | Cogent            | I    | ISP |
| 701   | Verizon           | I    | ISP |
| 1299  | Arelion           | I    | ISP |
| 3257  | GTT               | I    | ISP |
| 3491  | PCCW              | I    | ISP |
| 5511  | Orange            | I    | ISP |
| 6453  | TATA              | I    | ISP |
| 3320  | Deutsche Telekom  | I    | ISP |
| 6461  | Zayo              | I    | ISP |
| 6762  | Telecom Italia    | I    | ISP |
| 6830  | Liberty Global    | I    | ISP |
| 12956 | Telefonica        | I    | ISP |
| 15133 | Edgecast          | D    | CDN |
| 20940 | Akamai            | D    | CDN |
| 714   | Apple             | C    | Content |
| 2906  | Netflix           | C    | Content |
| 13335 | Cloudflare        | C    | Content |
| 15169 | Google            | C    | Content |

Count router-level links between every pair, reorder the ASes so heavily-interconnected ones sit
next to each other, and plot the result as a heatmap with each AS's ISP/CDN/Content category
marked along the top -- the single visual Q3 asks you to interpret.

In [ ]:
# ASN -> (name, category code) static table, matching the markdown table above.
AS_INFO = [
    (174,   "Cogent",            "I", "ISP"),
    (701,   "Verizon",           "I", "ISP"),
    (1299,  "Arelion",           "I", "ISP"),
    (3257,  "GTT",               "I", "ISP"),
    (3491,  "PCCW",              "I", "ISP"),
    (5511,  "Orange",            "I", "ISP"),
    (6453,  "TATA",              "I", "ISP"),
    (3320,  "Deutsche Telekom",  "I", "ISP"),
    (6461,  "Zayo",              "I", "ISP"),
    (6762,  "Telecom Italia",    "I", "ISP"),
    (6830,  "Liberty Global",    "I", "ISP"),
    (12956, "Telefonica",        "I", "ISP"),
    (15133, "Edgecast",          "D", "CDN"),
    (20940, "Akamai",            "D", "CDN"),
    (714,   "Apple",             "C", "Content"),
    (2906,  "Netflix",           "C", "Content"),
    (13335, "Cloudflare",        "C", "Content"),
    (15169, "Google",            "C", "Content"),
]
as_info_df = pd.DataFrame(AS_INFO, columns=["asn", "name", "code", "description"]).set_index("asn")
ASES, AS_CODE = as_info_df["name"].to_dict(), as_info_df["code"].to_dict()
ASES_SORTED = list(ASES.keys())

# Worked: count router-level links between each pair of the 18 ASes.
asn_sql = ", ".join(str(a) for a in ASES_SORTED)
pair_df = pd.read_sql(
    f"""
    WITH le_as AS (
        -- one row per (link, AS): DISTINCT collapses the grain (many endpoints/ASNs per
        -- link) so the self-join below does not fan out.
        SELECT DISTINCT le.link_id, na.asn
        FROM caida_itdk.itdk_link_endpoints le
        JOIN caida_itdk.itdk_node_as na ON na.node_id = le.node_id
        WHERE na.asn IN ({asn_sql})
    )
    -- self-join on link_id: a.asn < b.asn keeps one direction, so each unordered pair
    -- counts once.
    SELECT a.asn AS asn1, b.asn AS asn2, COUNT(*) AS link_count
    FROM le_as a
    JOIN le_as b ON b.link_id = a.link_id AND a.asn < b.asn
    GROUP BY a.asn, b.asn
    ORDER BY link_count DESC
    """,
    engine,
)
pair_df["name1"] = pair_df["asn1"].map(ASES)
pair_df["name2"] = pair_df["asn2"].map(ASES)
print(f"AS pairs with at least one router-level link: {len(pair_df)}")
print(pair_df[["name1", "name2", "link_count"]].head(20).to_string(index=False))

In [ ]:
%matplotlib inline
# YOUR CODE HERE
# Output: a log-scaled heatmap of pair_df, ASes reordered so heavily-interconnected ones sit
#         together, with each AS's category code (I/D/C) labeled along the top axis.
# Steps:
#   1. Reorder the 18 ASes with scipy.optimize.quadratic_assignment: build a symmetric
#      link-count matrix W (n x n, indexed like ASES_SORTED) and a position-distance matrix
#      D[p,q] = |p-q|; quadratic_assignment(W, D, method="faq") finds a permutation
#      minimizing sum_ij W[i,j] * D[pos_i, pos_j] -- i.e. puts well-linked AS pairs close
#      together on the axis. A single default-start call can land in a mediocre local
#      optimum, so run several randomized-start restarts (a fixed rng seed keeps this
#      reproducible) and keep the lowest-objective result.
#   2. Build a DataFrame `matrix` (18x18, indexed/columned by the QAP order's AS names),
#      filling both matrix.loc[n1,n2] and matrix.loc[n2,n1] from pair_df so it's symmetric;
#      set the zero (unconnected) entries to NaN so they render blank.
#   3. Plot with ax.imshow(np.log1p(matrix.values), cmap="YlOrRd"), axis tick labels from the
#      QAP-ordered AS names, and a secondary top x-axis showing each column's category code.
# student_code_start ------------------------------------------------
# YOUR CODE HERE
# student_code_end --------------------------------------------------

**Q3** Write two paragraphs explaining what the heatmap shows about router-level interconnection
among these 18 ASes. Which categories of AS are most interconnected, and which are least? Are
there ASes whose router-level behavior does not match their category label (a "content" AS that
interconnects like a transit backbone, say)? What does the overall pattern say about the
economics and architecture of the Internet?

*Your answer for Q3:*

Replace this line with your answer.

---

# Part 2 - Use the four provided tools through an agent (Q4-Q5)

The MCP server already ships four complete, working tools: `get_link_endpoints`,
`find_nodes_by_asn`, `search_nodes_by_geolocation`, and `lookup_router_hostnames`. Nothing needs
implementing before you can use them.

Part 2 runs **two fixed prompts** verbatim through `call_agent_with_mcp_tools`, seeded on
**AS15133 (Edgecast)** -- a real AS with exactly two geolocated router nodes in this snapshot,
small enough that every claim in the trace is checkable by hand. Your work is the audit: never
accept a number the agent reports without re-deriving it from a CSV you captured yourself. The
agent's transcript is a claim; the CSV is the evidence.

In [ ]:
# Supplied: the four tool contracts an agent sees when it connects. Keep this output
# with your submission so your grader knows exactly what the agent had to work with.
provided_tools = {tool["name"]: tool for tool in await list_itdk_tools()}
print("tools advertised to the agent:", sorted(provided_tools))
for name in sorted(provided_tools):
    print(json.dumps(provided_tools[name], indent=2, sort_keys=True))

### Fixed prompt A - ASN to geolocation

In [ ]:
PROMPT_A = (
    "Which router nodes does this snapshot assign to ASN 15133, where is each of those "
    "nodes geolocated, and which inference method produced each AS assignment and each "
    "location? Use the ITDK MCP tools; report the exact tool calls, arguments, and row "
    "counts you used."
)
trace_a, answer_a = await show_agent_run(PROMPT_A)

### Fixed prompt B - hostname vs. geolocation provenance

In [ ]:
PROMPT_B = (
    "For the router nodes assigned to ASN 15133, find the PTR hostname of each node's "
    "known interface and compare any location hint embedded in that hostname against the "
    "recorded geolocation for nodes in the US. State which method produced each location "
    "and whether the hostname is independent evidence."
)
trace_b, answer_b = await show_agent_run(PROMPT_B)

In [ ]:
# YOUR CODE HERE
# Output: q4_evidence (a list of evidence records) and q4_verification, a DataFrame with
#         one row per factual claim the agent made in the run you chose, columns:
#         claim, agent_value, verified_value, evidence_source, agrees.
# Steps:
#   1. Pick trace_a/answer_a or trace_b/answer_b and list the factual claims in it.
#   2. Re-run the calls each claim depends on with await call_itdk_tool(...), appending
#      every returned record to q4_evidence.
#   3. Load each result with load_tool_csv(...) and compute the number yourself; do not
#      copy the agent's number across. Note in particular whether the agent could resolve
#      each node's geolocation at all -- none of the four tools accept a node_id for that.
#   4. Also check the trace itself: did the agent call the tools it says it called, with
#      the arguments it says it used?
q4_evidence = []
q4_verification = None
q4_verification

**Q4** Pick one of the two fixed prompts. State its tool-call trace and the agent's answer, then
verify every factual claim against your own repeat calls. Report which claims held, which did not,
and the exact evidence (tool, arguments, artifact, row count) behind each verdict.

*Your answer for Q4:*

Replace this line with your answer.

In [ ]:
# YOUR CODE HERE
# Output: agent_process_log, a DataFrame with one row per tool call the agent made across
#         BOTH fixed prompts, columns:
#         step, prompt, tool, arguments, rows_returned, necessary, comment.
# Steps:
#   1. Transcribe from trace_a and trace_b -- this is the agent's behaviour, so record it
#      exactly as it happened.
#   2. In `necessary`, mark each call yes/no: needed, redundant with an earlier call, or a
#      guess made because no tool could answer the actual question (e.g. enumerating
#      countries one at a time to find where a specific node is located)?
#   3. In `comment`, note any assumption the agent stated without a supporting call.
agent_process_log = None
agent_process_log

**Q5** Critique the agent's *process* across both runs, not just its answers. Identify at least one
redundant call (same tool and arguments when the result was already available) or one unverified
assumption (a location, relationship, or completeness claim no tool result established), and say
what a more careful trace would have looked like. What would you change about the tool
descriptions -- or what tool is simply missing -- so a future agent makes fewer of those mistakes?

*Your answer for Q5:*

Replace this line with your answer.

---

# Part 3 - Build three new general-purpose tools, then investigate with all seven (Q6-Q9)

Parts 1 and 2 showed you what the four provided tools cannot do: none of them can look up a
node's location by `node_id`, find the router-level links between two specific ASes, or summarize
one AS's geolocated footprint. Now you add three tools that fill exactly those gaps.

**All the implementation happens in one file, `src/itdk_mcp/student_tools.py`** -- schema,
description, and fixed query for all three tools. `mcp_tools.py` merges your registries and routes
the calls automatically; you do not edit the completed server code. Add your tests to
`tests/test_student_tools.py` and remove its module-level `xfail` marker when the tools work.

| tool | argument(s) | returned columns | ordering |
| --- | --- | --- | --- |
| `get_node_geolocation` | `node_id` (identifier string) | `node_id, continent, country, region, city, latitude, longitude, method` | `node_id` |
| `find_router_links_between_asns` | `asn_a`, `asn_b` (integers) | `link_id, node_a, node_b` | `link_id, node_a, node_b` |
| `count_nodes_by_asn_and_country` | `asn` (integer) | `country, node_count` | `node_count` DESC, `country` |

These tools are deliberately **general-purpose**: each takes only a `node_id` or one/two `asn`
values, so an agent can combine them freely rather than being limited to one fixed question per
tool. `find_router_links_between_asns` in particular generalizes Part 1's Q1 border-router query
to any AS pair, and accepts `asn_a == asn_b` to mean "that AS's own intra-network links" rather
than treating it as an error. The full contracts and reference SQL are in
[ASSIGNMENT.md](ASSIGNMENT.md) section 4.

### Q6 - `get_node_geolocation`, end to end

In [ ]:
# YOUR CODE HERE
# Output: geo_contract (the discovered tool definition), geo_evidence and geo_row for one
#         Edgecast node (from find_nodes_by_asn(15133)), geo_invalid (a raw record whose
#         result.isError is true), and student_test_summary.
# Hint 1: rediscover the tool list after implementing the tool; the contract must show one
#         required node_id and additionalProperties false.
# Hint 2: call it for one of the two Edgecast node_ids with call_itdk_tool, load the CSV, and
#         compare the row against what you saw in SQL for that same node_id in Part 1.
# Hint 3: for the invalid call use call_itdk_raw so the error record is preserved (for
#         example a missing node_id, or an extra property like {"format": "geojson"}).
# Hint 4: student_test_summary is a DataFrame with one row per new tool, columns:
#         tool, test, layer, protected_failure -- cite the real test names you added to
#         tests/test_student_tools.py, the layer each sits at (discovery/schema,
#         validation/dispatch, or query contract), and the concrete regression each
#         catches (an interpolated identifier, a wrong projection, an unstable ORDER BY,
#         a missing bound parameter).
geo_contract = None
geo_evidence = None
geo_row = None
geo_invalid = None
student_test_summary = None

**Q6** Give the advertised contract for `get_node_geolocation` (discovered name, description,
input schema, output schema) and show one valid and one invalid call. Which layer rejects the
invalid call, and how do you know the invalid arguments never reached the query? Does the returned
row match what you saw in SQL in Part 1 for the same node? Finally, in one line: which of the
tests you added is the most useful, and what concrete failure would it catch?

*Your answer for Q6:*

Replace this line with your answer.

### Q7 - `find_router_links_between_asns` and its tradeoffs

In [ ]:
# YOUR CODE HERE
# Output: cogent_df and arelion_df -- find_router_links_between_asns(15133, 174) and
#         (15133, 1299) respectively -- plus intra_df, the same tool called with
#         asn_a = asn_b = 15133.
# Hint 1: look for a link_id that appears more than once in arelion_df with different
#         node_b values -- that is a single multi-endpoint "hyperlink" contributing more
#         than one (node_a, node_b) row, not a duplicate to be dropped.
# Hint 2: intra_df should return Edgecast's own intra-network router-level links (both
#         endpoints assigned to AS15133, if any exist) rather than an error -- explain in
#         your Q7 answer why that is the right behaviour for this tool's general contract.
cogent_df = None
arelion_df = None
intra_df = None

**Q7** Explain the three design choices in `find_router_links_between_asns`: staging each AS's
node set as a CTE before the self-join, allowing `asn_a == asn_b` rather than rejecting it, and
*not* deduplicating rows from a link with more than two endpoints. What does each choice buy and
what does each cost? Show the concrete rows in `arelion_df` (or another pair, if this one does
not exhibit it) that demonstrate a single link contributing more than one row. Then explain, using
`intra_df`, why `asn_a == asn_b` is a legitimate question rather than a degenerate one.

*Your answer for Q7:*

Replace this line with your answer.

### `count_nodes_by_asn_and_country` (no separate question -- you need it for Q8)

The third tool has no question of its own, but Q8 depends on it. Verify it works and compare its
total against `find_nodes_by_asn`'s row count to see how much geolocation coverage its INNER join
drops; the tradeoff is required reading in [ASSIGNMENT.md](ASSIGNMENT.md) section 4.

In [ ]:
# YOUR CODE HERE
# Output: edgecast_country_evidence and edgecast_country_df -- count_nodes_by_asn_and_country
#         for asn 15133 -- plus coverage_note: a one-line string comparing
#         edgecast_country_df["node_count"].sum() against find_nodes_by_asn(15133)'s row_count.
# Hint: for AS15133 the two totals should match (both of its nodes are geolocated in this
#       snapshot) -- pick a second, larger AS of your choice and show the same comparison for
#       one where they do NOT match, to demonstrate the INNER-join coverage gap concretely.
edgecast_country_evidence = None
edgecast_country_df = None
coverage_note = None

### Investigate with all seven tools

The next two cells re-run the same in-notebook agent pattern from Part 2, now that your server
advertises seven tools, still seeded on AS15133 (Edgecast) paired against AS174 (Cogent) and
AS1299 (Arelion). Run them, keep the traces, then answer Q8 and Q9 from evidence you capture
yourself.

In [ ]:
PROMPT_C = (
    "For ASN 15133, list every router node assigned to it, the assignment method for each, "
    "and each node's geolocation. Then, for AS15133 paired with AS174 (Cogent) and again "
    "with AS1299 (Arelion), report every router-level link between them. State the row "
    "count of every tool call you make."
)
trace_c, answer_c = await show_agent_run(PROMPT_C)

In [ ]:
PROMPT_D = (
    "Starting from ASN 15133, connect what you can: its router nodes, their geolocations, "
    "their PTR hostnames, and every router-level link they share with AS174 or AS1299. "
    "Then give a short interpretation of what this evidence does and does not establish."
)
trace_d, answer_d = await show_agent_run(PROMPT_D)

In [ ]:
# YOUR CODE HERE
# Output: edgecast_nodes_df (find_nodes_by_asn(15133) loaded), edgecast_geo_df (each node's
#         geolocation via get_node_geolocation), comparison_asn (the second AS you chose for
#         part3-3-code's coverage comparison), and link_care_notes -- a list of strings
#         flagging any record needing special care (a link with other than two endpoint
#         rows, a node with an AS assignment but no geolocation row, an interface with no
#         PTR record).
edgecast_nodes_df = None
edgecast_geo_df = None
comparison_asn = None
link_care_notes = []

**Q8** For seed ASN `15133`: how many distinct router nodes are returned and which assignment
methods occur? Explain why this is a snapshot-specific assignment count rather than a complete
current inventory of the AS. Then, using `count_nodes_by_asn_and_country`, compare AS15133's
geolocated footprint against one other AS of your choice, and identify which records need special
care -- a link with other than two endpoint rows, a node with an AS assignment but no geolocation
row, or an interface with no PTR record.

*Your answer for Q8:*

Replace this line with your answer.

In [ ]:
# YOUR CODE HERE
# Output: topology_evidence_table joining nodes, geolocations, hostnames, and cross-AS links
#         where the tools support it, starting from ASN 15133; and agent_claim_audit, one row
#         per factual claim in answer_d with columns:
#         claim, status, evidence_artifact, correction.
#         status is one of supported / underspecified / overstated / unsupported.
# Hint 1: merge only on documented keys (node_id, asn, ip); preserve unmatched rows rather
#         than manufacturing a link.
# Hint 2: keep every call record in topology_call_evidence (tool, arguments, artifact, row
#         count, columns, transformation applied afterwards).
topology_call_evidence = []
topology_evidence_table = None
agent_claim_audit = None

**Q9** Present your evidence table from the ASN 15133 seed and your audit of the fixed prompt D
answer. Mark each factual claim supported / underspecified / overstated / unsupported with the
exact artifact behind the verdict, and identify at least one claim that is overstated, unsupported,
or missing an essential limitation -- then correct it. If every claim is supported, name the most
important omitted limitation instead of inventing an error.

*Your answer for Q9:*

Replace this line with your answer.

---

Before submission, restart the kernel and run all cells. Confirm that the Part 2 and Part 3 cells
contain no direct database imports, connection strings, or SQL -- the direct SQL in Part 1 against
the real teaching snapshot is the one intentional exception -- and that every answer cites tool
arguments, artifact metadata, row counts, transformations, and limitations. Confirm that no API
key, bearer key, database password, or other credential appears in any cell or output.

[README](README.md) | [Assignment](ASSIGNMENT.md) | [Fixture data](data/README.md) | Notebook